# Setup Inicial

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('dataset_evasao_escolar_500_registros_com_valores_faltantes.csv')

In [ ]:
def generate_gaussian_distribution(series, x_label, y_label, title):
    sns.histplot(series, bins=20, kde=True)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.show()

In [ ]:
def get_invalid_numbers(series):
    numeric = pd.to_numeric(series, errors="coerce")
    invalids = series[numeric.isna() & series.notna()]
    return invalids

# Diagnóstico da Qualidade dos Dados

#### a) Verifique a existência de valores ausentes, apresentando a quantidade e o percentual de valores ausentes em cada atributo;

In [ ]:
null_summary = pd.DataFrame({
    "Qtd. Ausentes": df.isna().sum(),
    "% Ausentes": df.isna().mean() * 100
})

print(null_summary)

#### b) Verifique a existência de duplicidades;

In [ ]:
print(df.describe()["id_aluno"])

Há 30 ids repetidos, sendo eles: 

In [ ]:
ids_repetidos = df.loc[
    df["id_aluno"].duplicated(), 'id_aluno'
].unique().tolist()

print(ids_repetidos)

#### c) Verifique a existência de Inconsistências, identificando em seu relatório, pelo menos, três atributos que apresentem diferentes formas de representar a mesma informação. Exemplo: Maceio, maceio, MACEIÓ, Maceió.

| Problema | Atributo | Explicação |
|---|---|---|
| **Consistência** | **Idade** | Duas linhas fogem do padrão: `19 anos` e `vinte`. |
| **Consistência** | **Escola Origem** | 7 valores únicos, embora existam apenas 2 classificações. |
| **Consistência** | **Município Residência** | 9 valores únicos, mas apenas 6 municípios. |
| **Consistência** | **Pessoas Domicílio** | Um valor está escrito por extenso. |
| **Consistência** | **Acesso Internet** | 6 valores únicos para apenas `Sim` ou `Não`. |
| **Consistência** | **Trabalha** | Existem diferentes representações da mesma classificação. |
| **Consistência** | **Horas Estudo Semana** | Alguns valores incluem a palavra `horas`. |
| **Consistência** | **Frequência Percentual** | Diferentes formatos de representação da porcentagem. |
| **Consistência** | **Nota Média** | Um valor está escrito por extenso. |
| **Consistência** | **Reprovações** | Um valor está escrito por extenso. |
| **Consistência** | **Data Matrícula** | Mistura `2026-02-17` e `02/17/2026`. |
| **Consistência** | **Situação** | `Evadido` e `Evadiu` representam a mesma situação. |


#### d) Identifique valores inválidos ou suspeitos, que não estejam de acordo com o domínio esperado. Exemplos: idade = -3, frequência = 120, nota = 15.

| Problema | Atributo | Explicação |
|---|---|---|
| **Validade** | **Idade** | Uma linha apresenta o valor `145`. |
| **Validade** | **Renda Familiar Mensal** | Uma linha apresenta um valor negativo. |
| **Validade** | **Frequência Percentual** | Há linhas com valores acima de `100` e uma linha com valor negativo. |
| **Validade** | **Nota Média** | Há valores acima de `10` e um valor negativo. |

# Limpeza e Transformação

#### a) Correção/remoção de duplicidades – apresente um relatório informando a quantidade que havia antes, a quantidade removida e o total de registros após a operação de limpeza.

In [ ]:
df = df.drop_duplicates()

print(df.describe()["id_aluno"])

Com o código acima, removemos as 30 duplicatas identificadas anteriormente.

#### b) Realize a padronização de atributos categóricos que possuam diferenças de letras maiúsculas/minúsculas, espaços, acentuação, abreviações ou representações diferentes.

In [ ]:
col = df["escola_origem"].astype("string").str.strip().str.upper()

df["escola_origem"] = col.replace({
    "PUB.": "Pública",
    "PUBLICA": "Pública",
    "ESCOLA PÚBLICA": "Pública",
    "ESCOLA PRIVADA": "Privada",
    "PARTICULAR": "Privada",
    "PÚBLICA": "Pública",
    "PRIVADA": "Privada"
})

In [ ]:
df["escola_origem"].unique()

In [ ]:
col = df["municipio_residencia"].astype("string").str.strip().str.upper()

df["municipio_residencia"] = col.replace({
    "MACEIO": "Maceió",
    "MACEIÓ": "Maceió",
    "MARECHAL DEODORO": "Marechal Deodoro",
    "ARAPIRACA": "Arapiraca",
    "PALMEIRA DOS INDIOS": "Palmeira dos Índios",
    "PALMEIRA DOS ÍNDIOS": "Palmeira dos Índios",
    "RIO LARGO": "Rio Largo",
    "UNIAO DOS PALMARES": "União dos Palmares",
    "UNIÃO DOS PALMARES": "União dos Palmares"
})

In [ ]:
df["municipio_residencia"].unique()

In [ ]:
col = df["acesso_internet"].astype("string").str.strip().str.upper()

df["acesso_internet"] = col.replace({
    "1": "Sim",
    "S": "Sim",
    "SIM": "Sim",
    "YES": "Sim",
    "NÃO": "Não",
    "N": "Não",
    "NO": "Não"
})

In [ ]:
df["acesso_internet"].unique()

In [ ]:
col = df["turno"].astype("string").str.strip().str.upper()

df["turno"] = col.replace({
    "MANHÃ": "Manhã",
    "M": "Manhã",
    "MANHA": "Manhã",
    "TARDE": "Tarde",
    "T": "Tarde",
    "NOITE": "Noite",
    "N": "Noite"
})

In [ ]:
df["turno"].unique()

In [ ]:
col = df["situacao"].astype("string").str.strip().str.upper()

df["situacao"] = col.replace({
    "ATIVO": "Ativo",
    "EVADIDO": "Evadido",
    "EVADIU": "Evadido",
    "DESISTENTE": "Evadido",
    "MATRICULADO": "Matriculado",
})

In [ ]:
df["situacao"].unique()

#### c) Verifique e corrija atributos numéricos armazenados como texto.

In [ ]:
df["idade"] = df["idade"].replace({
    "vinte": "20",
    "20 anos": "20",
    "19 anos": "19"
})
df["idade"] = pd.to_numeric(df["idade"], errors="coerce")
df = df.dropna(subset=["idade"])
df = df[(df["idade"] > 0) & (df["idade"] < 100)]
df["idade"] = df["idade"].astype(int)

In [ ]:
generate_gaussian_distribution(df["idade"], "Idade", "Frequência", "Distribuição da idade")

Como não são valores bem distribuídos, e nem a presença de outliers, optei por remover a os valores nulos.

In [ ]:
df["renda_familiar_mensal"] = (
    df["renda_familiar_mensal"]
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
)

df["renda_familiar_mensal"] = pd.to_numeric(df["renda_familiar_mensal"], errors="coerce")

In [ ]:
len(df["pessoas_domicilio"].unique())

In [ ]:
df["pessoas_domicilio"] = df["pessoas_domicilio"].replace({
    "quatro": "4",
})

df["pessoas_domicilio"] = pd.to_numeric(df["pessoas_domicilio"], errors="coerce")

In [ ]:
horas_estudo_semana = (
    df["horas_estudo_semana"]
        .str.replace("horas", "", regex=False)
        .str.replace("h", "", regex=False)
        .str.strip()
)

horas_estudo_semana = pd.to_numeric(df["horas_estudo_semana"], errors="coerce")

In [ ]:
generate_gaussian_distribution(horas_estudo_semana, "Horas de estudo", "Frequência", "Distribuição das horas de estudo")

Como os dados são numéricos e bem distribuídos, optarei por tratar os valores nulos de horas_estudo com uma inputação estatística, aplicando a média.

In [ ]:
round(horas_estudo_semana.mean(), 1)

In [ ]:
df["horas_estudo_semana"] = (
    df["horas_estudo_semana"]
        .str.replace("horas", "", regex=False)
        .str.replace("h", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
)

df["horas_estudo_semana"] = df["horas_estudo_semana"].fillna(8.4)

df["horas_estudo_semana"] = pd.to_numeric(df["horas_estudo_semana"], errors="coerce")

In [ ]:
frequencia_percentual = df["frequencia_percentual"].replace({
    "oitenta": "80",
    "85%": "85",
})

frequencia_percentual = pd.to_numeric(df["idade"], errors="coerce")

In [ ]:
generate_gaussian_distribution(frequencia_percentual, "Frequência percentual", "Frequência", "Distribuição da frequência percentual")

In [ ]:
frequencia_percentual.median()

Apesar do percentual de nulos para essa coluna ser de 3.6%, eu não gostaria de descartar os valores nulos, prefiro optar por uma imputação estatística, pois das 18 linhas com valores nulos para frequência percentual, 4 delas possuem evasão justificada, o que eu acho interessante para a análise. A frequência percentual não é bem distribuída, portanto, estarei aplicando a mediana nos valores nulos.

In [ ]:
df["frequencia_percentual"] = df["frequencia_percentual"].replace({
    "oitenta": "80",
    "85%": "85",
})

df["frequencia_percentual"] = df["frequencia_percentual"].fillna(20)

df["frequencia_percentual"] = pd.to_numeric(df["horas_estudo_semana"], errors="coerce")

In [ ]:
get_invalid_numbers(df["nota_media"])

In [ ]:
nota_media = df["nota_media"].replace({
    "dez": "10",
    "8,5": "8.5"
})

nota_media = pd.to_numeric(nota_media, errors="coerce")

generate_gaussian_distribution(nota_media, "Nota média", "Frequência", "Distribuição da nota média")

In [ ]:
round(nota_media.median(), 1)

A nota média é bem distribuída, porém tem alguns outliers acima de 10, então vamos aplicar a mediana para os valores nulos novamente.

In [ ]:
df["nota_media"] = df["nota_media"].replace({
    "dez": "10",
    "8,5": "8.5"
})

df["nota_media"] = df["nota_media"].fillna(7.2)

df["nota_media"] = pd.to_numeric(df["nota_media"], errors="coerce")

In [ ]:
df["nota_media"].isna().sum()

In [ ]:
get_invalid_numbers(df["reprovacoes"])

In [ ]:
df["reprovacoes"] = df["reprovacoes"].replace({
    "duas": "2",
})

df["reprovacoes"] = pd.to_numeric(df["reprovacoes"], errors="coerce")

In [ ]:
df["reprovacoes"].isna().sum()

#### d) Padronize data_matricula para um único formato de data. Além disso, valores que não representam datas válidas devem ser identificados.



Formatos possíveis

Ano - Mês - Dia
Dia - Mês - Ano

Dia / Mês / Ano
Ano / Mês / Dia

Dia / Mês / Ano (Formato Reduzido, 26 ao invés de 2026)

In [ ]:
def format_date(date):
    if pd.isna(date):
        return None
    
    parts = []
    day, month, year = None, None, None
    
    if '-' in date:
        parts = date.split('-')
        
    if '/' in date:
        parts = date.split('/')

    if len(parts[0]) == 4:
        day, month, year = parts[2], parts[1], parts[0]

        if int(month) > 12:
            month, day = day, month
        
        return f"{year}-{month}-{day}"

    if len(parts[2]) == 4:
        day, month, year = parts[0], parts[1], parts[2]

        if int(month) > 12:
            month, day = day, month
            
        return f"{year}-{month}-{day}"
    
    day, month, year = '2026', parts[1], parts[2]

    if int(month) > 12:
        month, day = day, month
        
    return f"{year}-{month}-{day}"

In [ ]:
df["data_matricula"] = df["data_matricula"].apply(format_date)

In [ ]:
converted = pd.to_datetime(
    df["data_matricula"],
    format="%Y-%m-%d",
    errors="coerce"
)

failed = df.loc[
    converted.isna() & df["data_matricula"].notna(),
    "data_matricula"
]

print(failed)